In [1]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

# We MUST use imblearn's Pipeline, not sklearn's, to handle oversampling safely inside CV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# =====================================================================
# 1. Load the dataset
# =====================================================================
df = pd.read_csv('plrx.txt', delimiter='\t', header=None)
X = df.iloc[:, :12].values
y = df.iloc[:, 12].values

# =====================================================================
# 2. Define the Cross-Validation strategy
# =====================================================================
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# =====================================================================
# 3. Build the Pipeline (Scale -> Oversample -> SVM)
# =====================================================================
# SMOTE generates synthetic minority samples. 
# You can swap SMOTE() with RandomOverSampler() if you prefer exact duplicates.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    # Using the strongest SVM hyperparameters found in your previous tests
    ('svm', SVC(kernel='poly', C=10, gamma=0.1, random_state=42)) 
])

# =====================================================================
# 4. Generate Cross-Validated Predictions
# =====================================================================
# cross_val_predict trains the pipeline on 9 folds (applying SMOTE) 
# and predicts on the 1 untouched fold, repeating 10 times.
y_pred = cross_val_predict(pipeline, X, y, cv=cv, n_jobs=-1)

# =====================================================================
# 5. Output the Confusion Matrix & Metrics
# =====================================================================
print("Confusion Matrix (10-Fold CV with SMOTE):")
print(confusion_matrix(y, y_pred))

print("\nClassification Report:")
print(classification_report(y, y_pred))

Confusion Matrix (10-Fold CV with SMOTE):
[[76 54]
 [30 22]]

Classification Report:
              precision    recall  f1-score   support

         1.0       0.72      0.58      0.64       130
         2.0       0.29      0.42      0.34        52

    accuracy                           0.54       182
   macro avg       0.50      0.50      0.49       182
weighted avg       0.59      0.54      0.56       182



In [2]:
"""
Comparative analysis of oversampling techniques for EEG classification.
Safely applies each sampler inside a 10-fold Stratified CV loop.
"""

import pandas as pd
import warnings

from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, SVMSMOTE, ADASYN, RandomOverSampler
from imblearn.combine import SMOTETomek, SMOTEENN

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

# =====================================================================
# 1. Load the dataset
# =====================================================================
df = pd.read_csv('plrx.txt', delimiter='\t', header=None)
X = df.iloc[:, :12].values
y = df.iloc[:, 12].values

# =====================================================================
# 2. Define the Cross-Validation strategy
# =====================================================================
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

# =====================================================================
# 3. Define the Samplers to Compare
# =====================================================================
samplers = {
    'RandomOverSampler (Baseline)': RandomOverSampler(random_state=RANDOM_STATE),
    'Standard SMOTE': SMOTE(random_state=RANDOM_STATE),
    'BorderlineSMOTE': BorderlineSMOTE(random_state=RANDOM_STATE),
    'SVMSMOTE': SVMSMOTE(random_state=RANDOM_STATE),
    'ADASYN': ADASYN(random_state=RANDOM_STATE),
    'SMOTETomek (Cleaned)': SMOTETomek(random_state=RANDOM_STATE),
    'SMOTEENN (Aggressively Cleaned)': SMOTEENN(random_state=RANDOM_STATE)
}

# =====================================================================
# 4. Evaluate Each Sampler
# =====================================================================
print("="*60)
print("OVERSAMPLING COMPARISON: 10-FOLD CV SVM")
print("="*60)

for name, sampler in samplers.items():
    # Build a fresh pipeline for each sampler
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('sampler', sampler),
        # Base model matching your best configuration
        ('svm', SVC(kernel='poly', C=10, gamma=0.1, random_state=RANDOM_STATE)) 
    ])
    
    # Generate predictions safely using cross-validation
    y_pred = cross_val_predict(pipeline, X, y, cv=cv, n_jobs=-1)
    
    # Calculate metrics
    bal_acc = balanced_accuracy_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)
    
    # Print results
    print(f"\n--- {name} ---")
    print(f"Balanced Accuracy: {bal_acc:.3f}")
    print("Confusion Matrix:")
    print(cm)
    
    # Uncomment the line below if you want the full precision/recall breakdown for every method
    # print(classification_report(y, y_pred))

print("\n" + "="*60)
print("Key to Confusion Matrix:")
print("[[True Relaxed    False Planning]")
print(" [False Relaxed   True Planning ]]")

OVERSAMPLING COMPARISON: 10-FOLD CV SVM

--- RandomOverSampler (Baseline) ---
Balanced Accuracy: 0.548
Confusion Matrix:
[[90 40]
 [31 21]]

--- Standard SMOTE ---
Balanced Accuracy: 0.504
Confusion Matrix:
[[76 54]
 [30 22]]

--- BorderlineSMOTE ---
Balanced Accuracy: 0.525
Confusion Matrix:
[[79 51]
 [29 23]]

--- SVMSMOTE ---
Balanced Accuracy: 0.533
Confusion Matrix:
[[96 34]
 [35 17]]

--- ADASYN ---
Balanced Accuracy: 0.485
Confusion Matrix:
[[66 64]
 [28 24]]

--- SMOTETomek (Cleaned) ---
Balanced Accuracy: 0.508
Confusion Matrix:
[[72 58]
 [28 24]]

--- SMOTEENN (Aggressively Cleaned) ---
Balanced Accuracy: 0.513
Confusion Matrix:
[[31 99]
 [11 41]]

Key to Confusion Matrix:
[[True Relaxed    False Planning]
 [False Relaxed   True Planning ]]


In [3]:
"""
GridSearchCV hyperparameter tuning across multiple oversampling techniques.
Safely optimizes SVM (C, gamma, kernel) inside a 10-fold Stratified CV loop for each sampler.
"""

import pandas as pd
import warnings

from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import confusion_matrix, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, SVMSMOTE, ADASYN, RandomOverSampler
from imblearn.combine import SMOTETomek, SMOTEENN

warnings.filterwarnings('ignore')

RANDOM_STATE = 42

# =====================================================================
# 1. Load the dataset
# =====================================================================
df = pd.read_csv('plrx.txt', delimiter='\t', header=None)
X = df.iloc[:, :12].values
y = df.iloc[:, 12].values

# =====================================================================
# 2. Define the Cross-Validation strategy
# =====================================================================
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

# =====================================================================
# 3. Define the Samplers to Compare
# =====================================================================
samplers = {
    'RandomOverSampler': RandomOverSampler(random_state=RANDOM_STATE),
    'Standard SMOTE': SMOTE(random_state=RANDOM_STATE),
    'BorderlineSMOTE': BorderlineSMOTE(random_state=RANDOM_STATE),
    'SVMSMOTE': SVMSMOTE(random_state=RANDOM_STATE),
    'ADASYN': ADASYN(random_state=RANDOM_STATE),
    'SMOTETomek': SMOTETomek(random_state=RANDOM_STATE),
    'SMOTEENN': SMOTEENN(random_state=RANDOM_STATE)
}

# Hyperparameter grid to search over for the SVM step
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'svm__kernel': ['rbf', 'poly', 'linear']
}

# =====================================================================
# 4. Loop Through Each Sampler and Run Grid Search
# =====================================================================
print("="*75)
print("OVERSAMPLING + HYPERPARAMETER TUNING COMPARISON")
print("="*75)

for name, sampler in samplers.items():
    print(f"\nTuning SVM parameters using {name}...")
    
    # Build the imblearn pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('sampler', sampler),
        ('svm', SVC(random_state=RANDOM_STATE)) 
    ])
    
    # Set up the GridSearch inside the loop
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='balanced_accuracy',  # Optimize specifically for the imbalanced minority class
        n_jobs=-1
    )
    
    # Run the grid search
    grid_search.fit(X, y)
    
    # Get the best estimator configurations
    best_pipeline = grid_search.best_estimator_
    best_params = grid_search.best_params_
    best_cv_score = grid_search.best_score_
    
    # Generate honest cross-validated predictions using the best found hyperparameters
    y_pred = cross_val_predict(best_pipeline, X, y, cv=cv, n_jobs=-1)
    
    # Calculate performance metrics
    final_bal_acc = balanced_accuracy_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)
    
    # Print the optimized results for this sampler
    print(f"  -> Best Params: {best_params}")
    print(f"  -> CV Balanced Accuracy: {best_cv_score:.3f}")
    print("  -> Optimized Confusion Matrix:")
    print(cm)

print("\n" + "="*75)
print("Key to Confusion Matrix:")
print("[[True Relaxed    False Planning]")
print(" [False Relaxed   True Planning ]]")
print("="*75)

OVERSAMPLING + HYPERPARAMETER TUNING COMPARISON

Tuning SVM parameters using RandomOverSampler...
  -> Best Params: {'svm__C': 10, 'svm__gamma': 0.1, 'svm__kernel': 'poly'}
  -> CV Balanced Accuracy: 0.549
  -> Optimized Confusion Matrix:
[[90 40]
 [31 21]]

Tuning SVM parameters using Standard SMOTE...
  -> Best Params: {'svm__C': 1, 'svm__gamma': 1, 'svm__kernel': 'rbf'}
  -> CV Balanced Accuracy: 0.539
  -> Optimized Confusion Matrix:
[[125   5]
 [ 46   6]]

Tuning SVM parameters using BorderlineSMOTE...
  -> Best Params: {'svm__C': 100, 'svm__gamma': 'auto', 'svm__kernel': 'poly'}
  -> CV Balanced Accuracy: 0.542
  -> Optimized Confusion Matrix:
[[76 54]
 [26 26]]

Tuning SVM parameters using SVMSMOTE...
  -> Best Params: {'svm__C': 10, 'svm__gamma': 'auto', 'svm__kernel': 'poly'}
  -> CV Balanced Accuracy: 0.546
  -> Optimized Confusion Matrix:
[[97 33]
 [34 18]]

Tuning SVM parameters using ADASYN...
  -> Best Params: {'svm__C': 1, 'svm__gamma': 1, 'svm__kernel': 'rbf'}
  -> CV B

In [ ]:
import pandas as pd
import warnings

from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

warnings.filterwarnings('ignore')

RANDOM_STATE = 42

df = pd.read_csv('plrx.txt', delimiter='\t', header=None)
X = df.iloc[:, :12].values
y = df.iloc[:, 12].values


cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)


pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
    ('svm', SVC(random_state=RANDOM_STATE)) 
])


param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'svm__kernel': ['rbf', 'poly', 'linear']
}


print("Tuning SVM parameters using RandomOverSampler...")

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='balanced_accuracy',  # Focuses on optimizing both classes fairly
    n_jobs=-1
)

grid_search.fit(X, y)

best_pipeline = grid_search.best_estimator_
best_params = grid_search.best_params_
best_cv_score = grid_search.best_score_


y_pred = cross_val_predict(best_pipeline, X, y, cv=cv, n_jobs=-1)

final_bal_acc = balanced_accuracy_score(y, y_pred)
cm = confusion_matrix(y, y_pred)

print("\n" + "="*60)
print("FINAL OPTIMIZED RANDOM OVERSAMPLING RESULTS")
print("="*60)
print(f"Best Hyperparameters: {best_params}")
print(f"Grid Search CV Balanced Accuracy: {best_cv_score:.3f}")
print(f"Final Out-of-Fold Balanced Accuracy: {final_bal_acc:.3f}")

print("\nFinal Confusion Matrix:")
print(cm)

print("\nFinal Classification Report:")
print(classification_report(y, y_pred, target_names=["1.0 (Relaxed)", "2.0 (Planning)"]))

Tuning SVM parameters using RandomOverSampler...

FINAL OPTIMIZED RANDOM OVERSAMPLING RESULTS
Best Hyperparameters: {'svm__C': 10, 'svm__gamma': 0.1, 'svm__kernel': 'poly'}
Grid Search CV Balanced Accuracy: 0.549
Final Out-of-Fold Balanced Accuracy: 0.548

Final Confusion Matrix:
[[90 40]
 [31 21]]

Final Classification Report:
                precision    recall  f1-score   support

 1.0 (Relaxed)       0.74      0.69      0.72       130
2.0 (Planning)       0.34      0.40      0.37        52

      accuracy                           0.61       182
     macro avg       0.54      0.55      0.54       182
  weighted avg       0.63      0.61      0.62       182



In [5]:
"""
Optimized Pipeline for EEG Classification using Nested 10-Fold Cross-Validation.
Ensures completely unbiased evaluation by placing the hyperparameter grid search
entirely INSIDE the outer cross-validation loop.
"""

import pandas as pd
import warnings

from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

warnings.filterwarnings('ignore')

RANDOM_STATE = 42

# =====================================================================
# 1. Load the dataset
# =====================================================================
df = pd.read_csv('plrx.txt', delimiter='\t', header=None)
X = df.iloc[:, :12].values
y = df.iloc[:, 12].values

# =====================================================================
# 2. Define the Nested CV Folds (10-Fold Outer, 10-Fold Inner)
# =====================================================================
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

# =====================================================================
# 3. Build the Base Pipeline
# =====================================================================
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
    ('svm', SVC(random_state=RANDOM_STATE)) 
])

# =====================================================================
# 4. Define the Hyperparameter Grid to search
# =====================================================================
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'svm__kernel': ['rbf', 'poly', 'linear']
}

# =====================================================================
# 5. Set up the Inner Grid Search
# =====================================================================
# This object represents the ENTIRE tuning process.
grid_search_estimator = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=inner_cv,
    scoring='balanced_accuracy',  
    n_jobs=-1
)

# =====================================================================
# 6. Execute Outer 10-Fold Cross-Validation
# =====================================================================
print("Running Nested 10-Fold Cross-Validation (Tuning + Evaluation)...")

# cross_val_predict will now split the data into 10 outer folds.
# For each fold, it passes the training data into GridSearchCV, which runs 
# ITS OWN 10-fold inner split to find the best params, then evaluates on the outer fold.
y_pred = cross_val_predict(grid_search_estimator, X, y, cv=outer_cv, n_jobs=-1)

# Fit once on the entire dataset just to discover what the overall best parameters look like
grid_search_estimator.fit(X, y)
best_params = grid_search_estimator.best_params_
best_cv_score = grid_search_estimator.best_score_

# Calculate final metrics from the outer loops
final_bal_acc = balanced_accuracy_score(y, y_pred)
cm = confusion_matrix(y, y_pred)

# =====================================================================
# 7. Print Final Reports
# =====================================================================
print("\n" + "="*60)
print("FINAL NESTED 10-FOLD OVERSAMPLING RESULTS")
print("="*60)
print(f"Overall Best Hyperparameters (Full Data): {best_params}")
print(f"Inner CV Best Balanced Accuracy: {best_cv_score:.3f}")
print(f"True Unbiased Outer Balanced Accuracy: {final_bal_acc:.3f}")

print("\nFinal Confusion Matrix (Unbiased 10-Fold Outer Predict):")
print(cm)

print("\nFinal Classification Report:")
print(classification_report(y, y_pred, target_names=["1.0 (Relaxed)", "2.0 (Planning)"]))

Running Nested 10-Fold Cross-Validation (Tuning + Evaluation)...

FINAL NESTED 10-FOLD OVERSAMPLING RESULTS
Overall Best Hyperparameters (Full Data): {'svm__C': 10, 'svm__gamma': 0.1, 'svm__kernel': 'poly'}
Inner CV Best Balanced Accuracy: 0.549
True Unbiased Outer Balanced Accuracy: 0.479

Final Confusion Matrix (Unbiased 10-Fold Outer Predict):
[[97 33]
 [41 11]]

Final Classification Report:
                precision    recall  f1-score   support

 1.0 (Relaxed)       0.70      0.75      0.72       130
2.0 (Planning)       0.25      0.21      0.23        52

      accuracy                           0.59       182
     macro avg       0.48      0.48      0.48       182
  weighted avg       0.57      0.59      0.58       182

